# Benchmarking tree ensemble formulations

*The first example has fewer than 2000 variables and 2000 constraints, so it
can be solved with a size-limited license. The reproduction of Mišić (2020) at
the end of the notebook needs a full license.*

Which formulation of a tree ensemble solves fastest depends on the ensemble
and on the optimization problem around it. This notebook defines a small
function that builds and solves a model with each formulation and reports the
model size and the build and solve times. The example is deliberately small;
replace the regression and the model with your own to compare the formulations
on your problem.

The formulations are described in the
[Mixed Integer Formulations](https://gurobi-optimization-gurobi-machine-learning.readthedocs-hosted.com/en/latest/user/mip-models.html)
page of the user guide, and the notebook *Tree Ensemble Formulations* shows
how they differ on a small example.

In [ ]:
import time

import gurobipy as gp
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.ensemble import GradientBoostingRegressor

from gurobi_ml import add_predictor_constr

FORMULATIONS = ["leaf", "misic", "misic_lazy", "ocean", "biggs_perakis"]

## The model to compare on

We use the same example as in *Tree Ensemble Formulations*: a gradient boosting
regressor fitted on the diabetes dataset of scikit-learn, and a model that
maximizes its prediction over the range of the data.

`build_model` takes the name of a formulation and returns a model ready to be
solved. This is the function to replace with your own model: the benchmark
below only needs a model built with the given formulation.

In [ ]:
X, y = load_diabetes(return_X_y=True)
feature_lb = X.min(axis=0)
feature_ub = X.max(axis=0)

regression = GradientBoostingRegressor(n_estimators=20, max_depth=3, random_state=0)
regression.fit(X, y)


def build_model(formulation):
    m = gp.Model()
    m.Params.OutputFlag = 0
    x = m.addMVar((1, X.shape[1]), lb=feature_lb, ub=feature_ub, name="x")
    y_pred = m.addMVar((1, 1), lb=-gp.GRB.INFINITY, name="y")
    add_predictor_constr(m, regression, x, y_pred, formulation=formulation)
    m.setObjective(y_pred.sum(), gp.GRB.MAXIMIZE)
    return m

## The benchmark function

`benchmark` builds and solves the model once per formulation and per value of
Gurobi's `Seed` parameter. Solving with several seeds shows how much the solve
time varies from run to run, which for MIPs can be large. Further Gurobi
parameters, for example `TimeLimit` or `Threads`, are passed as keyword
arguments.

Each row of the result is one solve. The times are measured in seconds, both
as wall-clock time and as CPU time of the Python process, which includes the
threads of Gurobi.

In [ ]:
def benchmark(build_model, formulations, seeds, **params):
    rows = []
    for formulation in formulations:
        for seed in seeds:
            wall, cpu = time.perf_counter(), time.process_time()
            m = build_model(formulation)
            m.update()
            build_wall = time.perf_counter() - wall
            build_cpu = time.process_time() - cpu

            for name, value in params.items():
                m.setParam(name, value)
            m.Params.Seed = seed
            wall, cpu = time.perf_counter(), time.process_time()
            m.optimize()
            rows.append(
                {
                    "formulation": formulation,
                    "seed": seed,
                    "variables": m.NumVars,
                    "binaries": m.NumBinVars,
                    "linear constraints": m.NumConstrs,
                    "indicator constraints": m.NumGenConstrs,
                    "build wall [s]": build_wall,
                    "build CPU [s]": build_cpu,
                    "solve wall [s]": time.perf_counter() - wall,
                    "solve CPU [s]": time.process_time() - cpu,
                    "nodes": m.NodeCount,
                    "status": m.Status,
                    "objective": m.ObjVal if m.SolCount > 0 else float("nan"),
                }
            )
            m.dispose()
    return pd.DataFrame(rows)

## Run it

We solve each formulation with three seeds.

In [ ]:
results = benchmark(build_model, FORMULATIONS, seeds=range(3))
results

The table below summarizes the solves of each formulation: the model size,
the median times and number of nodes, and the range of the objective values.

In [ ]:
summary = results.groupby("formulation", sort=False).agg(
    variables=("variables", "first"),
    binaries=("binaries", "first"),
    constraints=("linear constraints", "first"),
    indicators=("indicator constraints", "first"),
    build_wall=("build wall [s]", "median"),
    solve_wall=("solve wall [s]", "median"),
    solve_cpu=("solve CPU [s]", "median"),
    nodes=("nodes", "median"),
    objective_min=("objective", "min"),
    objective_max=("objective", "max"),
)
summary

## Reading the results

This model solves in a fraction of a second with every formulation, so the
times above cannot tell the formulations apart. Larger ensembles do: with
`n_estimators=200` and `max_depth=7`, the model has about 12,000 variables
and needs a full license. With an Intel Xeon E-2388G (8 cores, 16 threads)
and 128 GB of memory, `"misic"` and `"misic_lazy"` solved it in about 40
seconds on average, `"ocean"` in 2 minutes, `"leaf"` in 7 minutes and
`"biggs_perakis"` in 25 minutes. To compare them on your
problem:

- Replace `regression` and `build_model` with your own ensemble and model,
  including the constraints of your decision problem. The ranking of the
  formulations changes with the problem, and maximizing a prediction over a
  box is not representative of most applications.
- Solve with several seeds and compare medians: the solve time of a MIP can
  vary a lot between seeds.
- Pass the parameters you use in practice, for example
  `benchmark(build_model, FORMULATIONS, seeds=range(5), TimeLimit=600, Threads=4)`.
- Compare the objective values, not only the times. With the default
  `epsilon=0`, `"leaf"`, `"ocean"` and `"biggs_perakis"` may report a value that
  no input attains, as shown in *Tree Ensemble Formulations*.
- `"ocean"` and `"biggs_perakis"` require finite bounds on the input variables
  used in splits, and raise an error otherwise.

## Reproducing Mišić (2020)

The same `benchmark` function can reproduce published experiments. We repeat
two experiments of Mišić, *Optimization of Tree Ensembles*, Operations Research
68(5):1605–1624, 2020 ([doi:10.1287/opre.2019.1928](https://doi.org/10.1287/opre.2019.1928)),
where the `"misic"` formulation comes from: random forests trained on two UCI
datasets, and the drug design case study. These models are too large for a
size-limited license.

Solve times depend on the machine, the number of threads and the Gurobi
version, so compare the trends between formulations rather than the absolute
times of the paper.

### Random forests on the concrete and wine datasets

The paper trains random forests with the defaults of the R package
`randomForest`: a third of the features considered at each split, at least five
samples per leaf and no depth limit. It uses the concrete compressive strength
(Yeh, 1998) and red wine quality (Cortez et al., 2009) datasets of the UCI
repository, and maximizes the prediction over the range of the data for
forests of increasing size.

`fetch_openml` downloads the datasets from OpenML on the first run and caches
them. scikit-learn draws the seed of each tree in turn from `random_state`, so a
forest of `T` trees is made of the first `T` trees of a larger forest with the
same `random_state`, as in the paper.

We solve forests of 10 and 50 trees, which takes a few seconds per formulation.
The paper goes up to 500 trees: set `N_TREES = [10, 50, 100, 200, 500]` to
reproduce the whole range, and raise `TIME_LIMIT` (the largest forests can take
an hour or more).

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestRegressor

N_TREES = [10, 50]
TIME_LIMIT = 600


def load_concrete():
    frame = fetch_openml(data_id=4353, as_frame=True).frame
    return frame.iloc[:, :-1].to_numpy(dtype=float), frame.iloc[:, -1].to_numpy(
        dtype=float
    )


def load_wine():
    data = fetch_openml(data_id=40691, as_frame=True)
    return data.data.to_numpy(dtype=float), data.target.astype(float).to_numpy()


def misic_forest(X, y, n_trees):
    """Random forest with the defaults of R's randomForest, as in the paper."""
    return RandomForestRegressor(
        n_estimators=n_trees, max_features=1 / 3, min_samples_leaf=5, random_state=0
    ).fit(X, y)


def box_model(predictor, X, add_constraints=None):
    """Model builder maximizing the prediction over the range of the data."""

    def build(formulation):
        m = gp.Model()
        m.Params.OutputFlag = 0
        x = m.addMVar((1, X.shape[1]), lb=X.min(axis=0), ub=X.max(axis=0))
        pred_constr = add_predictor_constr(m, predictor, x, formulation=formulation)
        if add_constraints is not None:
            add_constraints(m, pred_constr)
        m.setObjective(pred_constr.output.sum(), gp.GRB.MAXIMIZE)
        return m

    return build


runs = []
for dataset, load in [("concrete", load_concrete), ("winequality-red", load_wine)]:
    X_data, y_data = load()
    for n_trees in N_TREES:
        forest = misic_forest(X_data, y_data, n_trees)
        runs.append(
            benchmark(
                box_model(forest, X_data), FORMULATIONS, seeds=[0], TimeLimit=TIME_LIMIT
            ).assign(dataset=dataset, trees=n_trees)
        )
forest_results = pd.concat(runs, ignore_index=True)
forest_results.pivot_table(
    index=["dataset", "trees"],
    columns="formulation",
    values="solve wall [s]",
    sort=False,
)

### Drug design

In its case study (Section 6), the paper maximizes the predicted activity of a
molecule with random forests of 100 trees, trained with the same parameters on
the Merck molecular activity data released by Ma et al., *Deep Neural Nets as a
Method for Quantitative Structure–Activity Relationships*, Journal of Chemical
Information and Modeling 55(2):263–274, 2015
([doi:10.1021/ci500747n](https://doi.org/10.1021/ci500747n)).

The data is not downloaded automatically. To run this part:

1. Download the Supporting Information of Ma et al. (2015) from the article
   page linked above, and check its terms of use.
2. Unzip it, and put the training files of the targets, such as
   `METAB_training_disguised.csv`, in a directory `merck` next to this
   notebook, or set the environment variable `MERCK_DATA` to their directory.

The cells below are skipped when the data is missing. We use the `METAB`
target, one of the smallest sets (1569 molecules with 4372 descriptors).

The paper also restricts the solutions to molecules close to the training
data, with its proximity constraint (11): for every training molecule, at most
a fraction `c` of the trees may send the solution to the leaf of that
molecule. The constraint is linear in the leaf variables, which every
formulation exposes in the `tree_leaves` attribute. We solve the problem
without the constraint and with `c = 0.1`.

In [ ]:
import os
from pathlib import Path

import numpy as np
from scipy import sparse

MERCK_DIR = Path(os.environ.get("MERCK_DATA", "merck"))
TARGET = "METAB"
merck_file = MERCK_DIR / f"{TARGET}_training_disguised.csv"
if not merck_file.exists():
    print(f"{merck_file} not found: skipping the drug design case study.")


def proximity_constraints(X, forest, c):
    """Mišić (2020), constraint (11): for every training point, at most a
    fraction c of the trees send the solution to the leaf of that point."""
    leaf_of = forest.apply(X)  # leaf of each training point in each tree

    def add(m, pred_constr):
        total = 0
        for t, (leaf_vars, nodes) in enumerate(pred_constr.tree_leaves):
            column = {int(node): j for j, node in enumerate(nodes)}
            # Leaves that no input can reach have no variable and count as 0.
            pairs = [
                (i, column[int(node)])
                for i, node in enumerate(leaf_of[:, t])
                if int(node) in column
            ]
            rows, cols = zip(*pairs)
            indicator = sparse.csr_matrix(
                (np.ones(len(rows)), (rows, cols)), shape=(len(X), len(nodes))
            )
            total = total + indicator @ leaf_vars[0, :]
        m.addConstr(total <= c * len(pred_constr.tree_leaves))

    return add

In [ ]:
if merck_file.exists():
    frame = pd.read_csv(merck_file)
    X_drug = frame.iloc[:, 2:].to_numpy(dtype=float)
    y_drug = frame["Act"].to_numpy(dtype=float)
    drug_forest = misic_forest(X_drug, y_drug, 100)

    runs = []
    for c in (None, 0.1):
        constraints = (
            None if c is None else proximity_constraints(X_drug, drug_forest, c)
        )
        runs.append(
            benchmark(
                box_model(drug_forest, X_drug, constraints),
                FORMULATIONS,
                seeds=[0],
                TimeLimit=TIME_LIMIT,
            ).assign(proximity="none" if c is None else f"c = {c}")
        )
    drug_results = pd.concat(runs, ignore_index=True)
    display(
        drug_results.pivot_table(
            index="proximity",
            columns="formulation",
            values="solve wall [s]",
            sort=False,
        )
    )

Copyright © 2023-2026 Gurobi Optimization, LLC